NoTL_model

In [ ]:
import os
import math
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# -------------------------------
# 1) Global Settings
# -------------------------------
torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# -------------------------------
# 2) Load RAMA Dataset
# -------------------------------
# Input data structure: X_target_RAMA [5,5,1,5,N] -> (H, W, T, C, N)
# C=5 corresponds to: ERA5ws, CCMPws, ERA5sp, ERA5t2m, ERA5blh
mat = sio.loadmat(r'D:\data\train_data.mat')
X_raw = mat['X_target_RAMA']           # Shape: [5,5,1,5,N]
y_all = mat['Y_target_RAMA'].ravel()   # Shape: [N,] (observed wind speed)
years = mat['year_target_RAMA'].ravel()# Shape: [N,] (corresponding years)

_, _, T, C, N = X_raw.shape
assert T == 1 and C == 5, "Input dimensions must match (H=5, W=5, T=1, C=5)"

# Reshape to [N, C, T, H, W] for model input
X = np.transpose(X_raw, (4, 3, 2, 0, 1)).astype(np.float32)
y_all = y_all.astype(np.float32)

# -------------------------------
# 3) Model Definition (Branch CNN + Transformer)
# -------------------------------
class BranchCNN(nn.Module):
    """Branch CNN for independent spatial feature extraction of each variable"""
    def __init__(self, out_dim=20):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
            nn.Conv2d(16, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
            nn.Flatten(), nn.Linear(16 * 5 * 5, out_dim), nn.ReLU()
        )
    
    def forward(self, x):
        return self.net(x)

class CNNTransformer(nn.Module):
    """Hybrid model: Branch CNN + Transformer for wind speed reconstruction"""
    def __init__(self, num_branches=5, d_model=20, nhead=4, num_layers=3):
        super().__init__()
        self.branches = nn.ModuleList([BranchCNN(d_model) for _ in range(num_branches)])
        
        # Transformer Encoder Layer
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            dim_feedforward=80, 
            dropout=0.1, 
            activation=nn.GELU()
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.layer_norm = nn.LayerNorm(d_model)
        self.fc = nn.Linear(d_model, 1)

    def _positional_encoding(self, x):
        """Add positional encoding to preserve sequence order (variable order)"""
        L, B, D = x.size()  # L=num_branches=5, B=batch_size, D=d_model=20
        position = torch.arange(L, device=x.device).unsqueeze(1)  # [L, 1]
        div_term = torch.exp(torch.arange(0, D, 2, device=x.device) * (-math.log(10000.0) / D))
        pe = torch.zeros(L, 1, D, device=x.device)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        return x + pe

    def forward(self, x):
        # Extract features from each branch (per variable)
        branch_features = []
        for i, branch in enumerate(self.branches):
            # x shape: [B, C, T, H, W] -> extract i-th variable: [B, 1, H, W]
            x_var = x[:, i, 0, :, :].unsqueeze(1)  # T=1, add channel dim
            branch_features.append(branch(x_var))  # [B, d_model]
        
        # Stack to sequence: [L=5, B, d_model]
        seq = torch.stack(branch_features, dim=0)
        seq = self._positional_encoding(seq)
        
        # Transformer encoding
        encoder_out = self.encoder(seq)  # [5, B, 20]
        encoder_out = self.layer_norm(encoder_out)
        
        # Predict wind speed (use the last sequence token)
        return self.fc(encoder_out[-1]).squeeze(1)  # [B]

# -------------------------------
# 4) Evaluation Metrics
# -------------------------------
def calculate_metrics(y_true, y_pred):
    """Calculate RMSE, MAE, PCC, R², MBE"""
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    pcc = np.corrcoef(y_true, y_pred)[0, 1] if len(np.unique(y_true)) > 1 else 0.0
    r2 = r2_score(y_true, y_pred)
    mbe = np.mean(y_pred - y_true)
    return rmse, mae, pcc, r2, mbe

# -------------------------------
# 5) Leave-One-Year-Out Cross Validation (LOOCV)
# -------------------------------
unique_years = np.unique(years)
K_folds = len(unique_years)
print(f"Total folds (years): {K_folds}, Years: {unique_years}")

# Create output directory
output_dir = r'D:\data\wind_speed_median\branch_cnn_transformer_notl'
os.makedirs(output_dir, exist_ok=True)

# Store fold-wise results
fold_results = []

for fold_idx, test_year in enumerate(unique_years, start=1):
    print(f"\n=== Fold {fold_idx}/{K_folds}: Leave-Out Year {test_year} ===")
    
    # Split train/validation/test sets
    test_mask = (years == test_year)
    X_test = X[test_mask]
    y_test = y_all[test_mask]
    X_remaining = X[~test_mask]
    y_remaining = y_all[~test_mask]
    
    # Split remaining data into train (80%) and validation (20%)
    permuted_idx = np.random.permutation(len(y_remaining))
    train_split = int(0.8 * len(permuted_idx))
    train_idx, val_idx = permuted_idx[:train_split], permuted_idx[train_split:]
    
    X_train = X_remaining[train_idx]
    y_train = y_remaining[train_idx]
    X_val = X_remaining[val_idx]
    y_val = y_remaining[val_idx]
    
    # Normalization (fit on training set only)
    train_mean = X_train.mean(axis=(0, 2, 3, 4), keepdims=True)
    train_std = X_train.std(axis=(0, 2, 3, 4), keepdims=True) + 1e-8  # Avoid division by zero
    
    X_train_norm = (X_train - train_mean) / train_std
    X_val_norm = (X_val - train_mean) / train_std
    X_test_norm = (X_test - train_mean) / train_std
    
    # Create DataLoaders
    def create_dataloader(x_data, y_data=None):
        if y_data is None:
            dataset = TensorDataset(torch.tensor(x_data, device=device))
        else:
            dataset = TensorDataset(
                torch.tensor(x_data, device=device),
                torch.tensor(y_data, device=device)
            )
        return DataLoader(dataset, batch_size=32, shuffle=(y_data is not None))
    
    train_loader = create_dataloader(X_train_norm, y_train)
    val_loader = create_dataloader(X_val_norm, y_val)
    test_loader = create_dataloader(X_test_norm, y_test)
    
    # Initialize model, optimizer, loss function
    model = CNNTransformer().to(device)
    
    # Freeze first Transformer layer (as per paper: layered freezing strategy)
    for param in model.encoder.layers[0].parameters():
        param.requires_grad = False
    
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.MSELoss()
    
    # Early stopping settings
    best_val_rmse = float('inf')
    patience = 0
    max_patience = 10
    best_model_state = None
    
    # Training loop
    for epoch in range(1, 51):
        model.train()
        train_loss = 0.0
        
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            predictions = model(batch_x)
            loss = criterion(predictions, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * batch_x.size(0)
        
        avg_train_loss = train_loss / len(train_loader.dataset)
        
        # Validation
        model.eval()
        val_predictions = []
        val_ground_truth = []
        
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                val_pred = model(batch_x)
                val_predictions.extend(val_pred.cpu().numpy())
                val_ground_truth.extend(batch_y.cpu().numpy())
        
        val_rmse, val_mae, val_pcc, val_r2, val_mbe = calculate_metrics(val_ground_truth, val_predictions)
        print(f"Epoch {epoch:02d} | Train Loss: {avg_train_loss:.4f} | Val RMSE: {val_rmse:.3f}, MAE: {val_mae:.3f}, PCC: {val_pcc:.3f}, R²: {val_r2:.3f}")
        
        # Update best model
        if val_rmse < best_val_rmse - 1e-4:
            best_val_rmse = val_rmse
            patience = 0
            best_model_state = model.state_dict().copy()
        else:
            patience += 1
            if patience >= max_patience:
                print(f"Early stopping triggered at epoch {epoch}")
                break
    
    # Evaluate on test set with best model
    model.load_state_dict(best_model_state)
    model.eval()
    test_predictions = []
    test_ground_truth = []
    
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            test_pred = model(batch_x)
            test_predictions.extend(test_pred.cpu().numpy())
            test_ground_truth.extend(batch_y.cpu().numpy())
    
    test_rmse, test_mae, test_pcc, test_r2, test_mbe = calculate_metrics(test_ground_truth, test_predictions)
    print(f"Fold {fold_idx} Test Results | RMSE: {test_rmse:.3f}, MAE: {test_mae:.3f}, PCC: {test_pcc:.3f}, R²: {test_r2:.3f}, MBE: {test_mbe:.3f}")
    
    # Save fold results
    fold_results.append({
        'year': test_year,
        'rmse': test_rmse,
        'mae': test_mae,
        'pcc': test_pcc,
        'r2': test_r2,
        'mbe': test_mbe
    })
    
    # Save test set predictions (obs vs pred)
    np.savez(
        os.path.join(output_dir, f'fold_{fold_idx}_year_{test_year}_obs_pred.npz'),
        year=test_year,
        observed=test_ground_truth,
        predicted=test_predictions,
        metrics={'rmse': test_rmse, 'mae': test_mae, 'pcc': test_pcc, 'r2': test_r2, 'mbe': test_mbe}
    )

# -------------------------------
# 6) Train Final Model on Full Dataset (2014-2020)
# -------------------------------
print("\n=== Training Final NoTL Model on Full Dataset (2014-2020) ===")
# Full dataset normalization
full_mean = X.mean(axis=(0, 2, 3, 4), keepdims=True)
full_std = X.std(axis=(0, 2, 3, 4), keepdims=True) + 1e-8
X_full_norm = (X - full_mean) / full_std

# Create full dataset loader
full_dataset = TensorDataset(torch.tensor(X_full_norm, device=device), torch.tensor(y_all, device=device))
full_loader = DataLoader(full_dataset, batch_size=32, shuffle=True)

# Initialize and train final model
final_model = CNNTransformer().to(device)
# Freeze first Transformer layer
for param in final_model.encoder.layers[0].parameters():
    param.requires_grad = False

final_optimizer = optim.Adam(final_model.parameters(), lr=1e-4)
final_criterion = nn.MSELoss()

# Train for 50 epochs (or early stopping)
for epoch in range(1, 51):
    final_model.train()
    epoch_loss = 0.0
    for batch_x, batch_y in full_loader:
        final_optimizer.zero_grad()
        pred = final_model(batch_x)
        loss = final_criterion(pred, batch_y)
        loss.backward()
        final_optimizer.step()
        epoch_loss += loss.item() * batch_x.size(0)
    
    avg_epoch_loss = epoch_loss / len(full_loader.dataset)
    print(f"Final Model Epoch {epoch:02d} | Loss: {avg_epoch_loss:.4f}")

# Save final model and normalization parameters
final_model_path = os.path.join(output_dir, 'final_notl_model.pth')
torch.save({
    'model_state_dict': final_model.state_dict(),
    'normalization': {'mean': full_mean, 'std': full_std},
    'model_config': {'num_branches': 5, 'd_model': 20, 'nhead': 4, 'num_layers': 3}
}, final_model_path)

# Save cross-validation summary
cv_summary = pd.DataFrame(fold_results)
cv_summary.to_csv(os.path.join(output_dir, 'loocv_results.csv'), index=False)
print(f"\nLOOCV Summary:\n{cv_summary.describe()}")
print(f"\nFinal model saved to: {final_model_path}")